In [1]:
import pandas as pd
import numpy as np

# 1. Load the processed data
df = pd.read_csv("../data/processed/wmt_pl_enriched.csv")

# 2. Simulate a Corporate Budget for 2026 based on 2025 Actuals
latest_actuals = df[df['FiscalYear'] == 2026].iloc[0]
prior_actuals = df[df['FiscalYear'] == 2025].iloc[0]

# Executive Targets for 2026
budget_2026 = {
    'Revenue': prior_actuals['Revenue'] * 1.05,
    'COGS': prior_actuals['COGS'] * 1.04, 
    'OperatingIncome': prior_actuals['OperatingIncome'] * 1.08 
}

# 3. Build the Budget vs. Actuals (BvA) DataFrame
variance_df = pd.DataFrame({
    'Metric': ['Revenue', 'COGS', 'OperatingIncome'],
    'Actuals_2026': [latest_actuals['Revenue'], latest_actuals['COGS'], latest_actuals['OperatingIncome']],
    'Budget_2026': [budget_2026['Revenue'], budget_2026['COGS'], budget_2026['OperatingIncome']]
})

# 4. Calculate Absolute Variance ($)
# For Revenue and Income: Higher Actuals = Good (Favorable)
# For Expenses (COGS): Higher Actuals = Bad (Unfavorable)
variance_df['Variance_$'] = variance_df['Actuals_2026'] - variance_df['Budget_2026']

# Invert the sign for COGS so a positive variance always means "Favorable to the business"
variance_df.loc[variance_df['Metric'] == 'COGS', 'Variance_$'] = variance_df['Budget_2026'] - variance_df['Actuals_2026']

# 5. Calculate Percentage Variance (%)
variance_df['Variance_%'] = (variance_df['Variance_$'] / variance_df['Budget_2026']) * 100

# 6. Flag Favorable (F) vs Unfavorable (U)
variance_df['Status'] = variance_df['Variance_$'].apply(lambda x: 'Favorable (F)' if x >= 0 else 'Unfavorable (U)')

# Clean up formatting for the executive view
for col in ['Actuals_2026', 'Budget_2026', 'Variance_$', 'Variance_%']:
    variance_df[col] = variance_df[col].round(2)

# Save the variance report to the interim folder
variance_df.to_csv("../data/interim/wmt_2026_variance.csv", index=False)

print("✅ Variance Analysis Complete.")
print("\n📊 2026 Budget vs. Actuals (BvA) Report (in $ Billions):")
print(variance_df.to_string(index=False))

✅ Variance Analysis Complete.

📊 2026 Budget vs. Actuals (BvA) Report (in $ Billions):
         Metric  Actuals_2026  Budget_2026  Variance_$  Variance_%          Status
        Revenue        713.16       715.03       -1.87       -0.26 Unfavorable (U)
           COGS        535.40       532.22       -3.18       -0.60 Unfavorable (U)
OperatingIncome         29.82        31.70       -1.88       -5.92 Unfavorable (U)
